## Project Introduction
As part of the data-engineering team at an e-commerce company specializing in **technology market insights**, your primary task is to design and implement an end-to-end **data pipeline** and **analysis workflow**. The company aims to analyze trends in the stock market for **prominent AI technology companies** to identify **market trends** and sell **actionable insights** to technology-focused businesses.

We chose to use **Yahoo Finance** as it provides reliable and extensive data on AI-related technology stocks, enabling:

1. Tracking stock performance over time.
2. Analyzing market trends and sector behavior.
3. Generating actionable insights for technology companies.

In [2]:
# Getting API access Yahoo Finance
import yfinance as yf
import numpy as np
import pandas as pd
import json
import pandas_gbq
from datetime import datetime
from google.cloud import bigquery
from google.cloud import storage
from io import StringIO

# Fetch list of ticker data from Yahoo Finance
tickers = ["MSFT", "GOOG", "META"]
tickersData = yf.download(tickers, start="2020-01-01", end="2025-05-30")



[*********************100%***********************]  3 of 3 completed


We chose **Goolge Cloud Storage**, as it is powerful and scalable solution for managing and storing data. GCS provides a scalable, secure, and cost-efficient platform that integrates seamlessly with your workflow, making it an excellent choice for storing and managing data

In [10]:
# Connect to GCS
GCSClient = storage.Client()
bucket = GCSClient.bucket("tickers-2020-2025")
blob = bucket.blob("tickersData.csv")


We chose to save the data in **CSV** format, as it provides:

1. Simplicity for data extraction and transformation.

2. Compatibility with BigQuery and other tools in your workflow.

3. Portability and ease of debugging.

In [16]:
# Save as CSV string
tickersDataCSV = tickersData.to_csv(index = True)

# Store data into GCS
blob.upload_from_string(tickersDataCSV, content_type='text/csv')

# Fetch tickers data from GCS
tickersDataCSV = blob.download_as_text()

In [17]:
# #Setup BigQuery Client
BQClient = bigquery.Client()

# #Transform data
tickersDataDF = pd.read_csv(StringIO(tickersDataCSV))
tickersDataDF.columns = [
    col.replace(".", "_") for col in tickersDataDF.columns  # Replace dots with underscores
]



# #Load DF into BigQuery
projectID = "stroff-1130"
datasetID = "NTUProj"
tableID = f"{projectID}.{datasetID}.tickers"


# Load DataFrame into BigQuery
try:
    pandas_gbq.to_gbq(
        tickersDataDF,
        destination_table=tableID,
        project_id=projectID,
        if_exists="replace"  # Options: 'fail', 'replace', 'append'
    )
    print(f"Data successfully loaded into {tableID}.")
except Exception as e:
    print(f"Error loading data into BigQuery: {e}")



100%|██████████| 1/1 [00:00<00:00, 313.76it/s]

Data successfully loaded into stroff-1130.NTUProj.tickers.
